# Dyadic Tomography Reverse Engine v2 — FIXED

## Fractional $\pi$ Seed + Finite-Cone FOLD-TOMO Geometry

**Corrected from:** `dyadic_tomography_reverse_engine_v1.pdf`  
**Branch:** `FOLD-TOMO`  
**Version:** v2 fixed

This notebook fixes the prototype issues:

1. skips the leading integer digit `3` of $\pi$;
2. uses the first $2048$ **fractional** digits;
3. uses the finite open cone, not circular wrap;
4. uses dyadic terminal rows with length $2^k$, not $2048$ cyclic rows;
5. excludes the identity level from terminal constraints;
6. verifies the corrected rank lock:

$$
\operatorname{rank}(C_{\mathrm{dyadic}})=1024,
$$

$$
\operatorname{rank}
\begin{bmatrix}
C_{\mathrm{dyadic}}\\
C_{448}
\end{bmatrix}
=1600,
$$

$$
2048-1600=448.
$$

Core thesis:

$$
\boxed{
\text{Terminal rows are checksum tomography, not debris.}
}
$$


In [ ]:
# Cell 1 — Environment setup

import sys
import subprocess
import importlib.util
from pathlib import Path
from collections import defaultdict
import math
import time
import random

required = ["numpy", "matplotlib", "mpmath"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

N = 2048
RUN_FULL_RANK = True
RUN_VISUALS = True

print("Dyadic Tomography Reverse Engine v2 — FIXED")
print("=" * 60)
print("Environment ready.")


## 1. Correct Seed: First 2048 Fractional Digits of $\pi$

The old prototype included the leading integer digit:

$$
3.1415\ldots \rightarrow 3,1,4,1,5,\ldots
$$

The corrected FOLD-TOMO seed skips the leading `3`:

$$
D^{(0)}=(1,4,1,5,9,\ldots).
$$


In [ ]:
# Cell 2 — Build corrected pi seed

mp.mp.dps = N + 50

pi_fractional = str(mp.pi).split(".")[1][:N]
seed_decimal = [int(d) for d in pi_fractional]
seed_parity = [d % 2 for d in seed_decimal]

print(f"Seed length: {len(seed_decimal)}")
print(f"First 50 fractional digits: {pi_fractional[:50]}")
print(f"Last 50 fractional digits:  {pi_fractional[-50:]}")
print(f"Decimal digit sum S0^(10): {sum(seed_decimal)}")
print(f"Parity Hamming weight S0^(2): {sum(seed_parity)}")
print(f"Parity residue R0: {sum(seed_parity) - N/2}")

distribution = sorted([(d, seed_decimal.count(d)) for d in range(10)], key=lambda x: -x[1])
print("Digit distribution:", distribution)

assert len(seed_decimal) == 2048
assert pi_fractional[:20] == "14159265358979323846"
assert sum(seed_decimal) == 9338
assert sum(seed_parity) == 1034
assert sum(seed_parity) - N/2 == 10.0

print("Correct fractional seed locked.")


## 2. Correct Fold Geometry: Finite Open Cone

No circular wrap.

Decimal fold:

$$
d_i^{(\ell+1)}
=
|d_{i+1}^{(\ell)}-d_i^{(\ell)}|.
$$

Parity shadow:

$$
x_i^{(\ell+1)}
=
x_i^{(\ell)}\oplus x_{i+1}^{(\ell)}.
$$

Row length:

$$
N_\ell=N-\ell.
$$


In [ ]:
# Cell 3 — Finite-cone fold builders

def ducci_fold_decimal_open(row):
    return [abs(row[i+1] - row[i]) for i in range(len(row) - 1)]

def xor_fold_open(row):
    return [row[i] ^ row[i+1] for i in range(len(row) - 1)]

def build_decimal_trajectory(seed):
    rows = [list(seed)]
    cur = list(seed)
    while len(cur) > 1:
        cur = ducci_fold_decimal_open(cur)
        rows.append(cur)
    return rows

def build_xor_trajectory(seed_bits):
    rows = [list(seed_bits)]
    cur = list(seed_bits)
    while len(cur) > 1:
        cur = xor_fold_open(cur)
        rows.append(cur)
    return rows

t0 = time.time()
trajectory_decimal = build_decimal_trajectory(seed_decimal)
trajectory_parity = build_xor_trajectory(seed_parity)
print(f"Built finite-cone trajectories in {time.time()-t0:.3f} seconds.")
print(f"Decimal trajectory levels: {len(trajectory_decimal)}")
print(f"Parity trajectory levels:  {len(trajectory_parity)}")
print(f"Level 0 length: {len(trajectory_parity[0])}")
print(f"Final level: {len(trajectory_parity)-1}, length: {len(trajectory_parity[-1])}, value: {trajectory_parity[-1]}")

assert len(trajectory_decimal) == 2048
assert len(trajectory_parity) == 2048
assert len(trajectory_parity[0]) == 2048
assert len(trajectory_parity[-1]) == 1


## 3. Exact Parity Shadow Verification

$$
|a-b|\bmod2=(a+b)\bmod2=a\oplus b.
$$


In [ ]:
# Cell 4 — Verify parity shadow equality

violations = 0
total_checks = 0
first_bad = None

for level in range(len(trajectory_decimal)):
    decimal_mod2 = [d % 2 for d in trajectory_decimal[level]]
    parity_direct = trajectory_parity[level]
    if decimal_mod2 != parity_direct and first_bad is None:
        first_bad = level
    for a, b in zip(decimal_mod2, parity_direct):
        if a != b:
            violations += 1
        total_checks += 1

print("Parity shadow verification:")
print(f"  Total checks: {total_checks:,}")
print(f"  Violations: {violations}")
print(f"  First bad level: {first_bad}")
print(f"  Status: {'VERIFIED' if violations == 0 else 'FAILED'}")

assert violations == 0


## 4. Glyph Reader Metrics

$$
N_\ell=N-\ell,
\qquad
S_\ell=\sum_i x_i^{(\ell)},
\qquad
\rho_\ell=\frac{S_\ell}{N_\ell},
\qquad
R_\ell=S_\ell-\frac{N_\ell}{2}.
$$


In [ ]:
# Cell 5 — Trace metrics

def trace_metrics(rows):
    N_vals = np.array([len(r) for r in rows], dtype=int)
    S_vals = np.array([sum(r) for r in rows], dtype=int)
    rho = S_vals / N_vals
    R = S_vals - N_vals / 2.0
    return {
        "N": N_vals,
        "S": S_vals,
        "rho": rho,
        "R": R,
        "dS": np.diff(S_vals),
        "dR": np.diff(R),
        "locks": [i for i, val in enumerate(R) if val == 0],
    }

binary_trace = trace_metrics(trajectory_parity)

decimal_S = np.array([sum(r) for r in trajectory_decimal], dtype=int)
amp_collapse_13 = (decimal_S[0] - decimal_S[13]) / decimal_S[0]
expected_amp = (9338 - 1092) / 9338

print("Decimal S0:", decimal_S[0])
print("Decimal S13:", decimal_S[13])
print("Amplitude collapse by level 13:", amp_collapse_13)
print()
print("Binary N0:", binary_trace["N"][0])
print("Binary S0:", binary_trace["S"][0])
print("Binary R0:", binary_trace["R"][0])
print("R=0 lock count:", len(binary_trace["locks"]))
print("First 15 locks:", binary_trace["locks"][:15])

assert decimal_S[0] == 9338
assert decimal_S[13] == 1092
assert abs(amp_collapse_13 - expected_amp) < 1e-15
assert binary_trace["S"][0] == 1034
assert binary_trace["R"][0] == 10.0
assert len(binary_trace["locks"]) == 44


## 5. Lucas Mask Law

$$
x^{(\ell)}=(I+E)^\ell x^{(0)}.
$$

Lucas theorem over $GF(2)$:

$$
\binom{\ell}{j}\equiv1\pmod2
\iff
j\ \&\ \sim\ell=0.
$$

Thus:

$$
x_i^{(\ell)}
=
\bigoplus_{j\subseteq\ell}
x_{i+j}^{(0)}.
$$


In [ ]:
# Cell 6 — Lucas masks

def popcount(n):
    return bin(int(n)).count("1")

def lucas_offsets(level):
    offsets = [0]
    bit = 1
    n = int(level)
    while n:
        if n & 1:
            offsets = offsets + [o + bit for o in offsets]
        n >>= 1
        bit <<= 1
    return offsets

def row_from_lucas(seed_bits, level):
    offsets = lucas_offsets(level)
    row_len = len(seed_bits) - level
    out = []
    for i in range(row_len):
        v = 0
        for j in offsets:
            v ^= seed_bits[i+j]
        out.append(v)
    return out

test_levels = [0, 1, 2, 3, 13, 110, 300, 342, 376, 448, 496, 512, 556, 576, 640, 688, 1024, 2032, 2040, 2044, 2046, 2047]

for level in test_levels:
    offsets = lucas_offsets(level)
    direct = row_from_lucas(seed_parity, level)
    assert len(offsets) == 2 ** popcount(level)
    assert direct == trajectory_parity[level], f"Lucas mismatch at level {level}"

for level in [448, 512, 576, 2047]:
    offsets = lucas_offsets(level)
    print(f"level={level:4d}, binary={level:b}, popcount={popcount(level)}, mask_size={len(offsets)}")
    print(" offsets:", offsets[:20], "..." if len(offsets) > 20 else "")
    print()

assert lucas_offsets(448) == [0,64,128,192,256,320,384,448]
assert lucas_offsets(512) == [0,512]
assert lucas_offsets(576) == [0,64,512,576]

print("Lucas mask law verified.")


## 6. Structural Pins: 448, 512, and Mark-9 576

448:

$$
M_{448}=\{0,64,128,192,256,320,384,448\},
$$

$$
N_{448}=1600,\quad S_{448}=800,\quad R_{448}=0.
$$

512:

$$
M_{512}=\{0,512\},
$$

$$
N_{512}=1536,\quad S_{512}=764,\quad R_{512}=-4.
$$

Mark-9 phase checkpoint:

$$
H=\frac{\pi}{9},\quad 9H=\pi,\quad \ell_9=9\cdot64=576.
$$

Since:

$$
576=512+64,
$$

$$
M_{576}=\{0,64,512,576\}.
$$

Verified:

$$
N_{576}=1472,\quad S_{576}=749,\quad R_{576}=13.
$$


In [ ]:
# Cell 7 — Verify structural pins

def summarize_level(level):
    row = trajectory_parity[level]
    n = len(row)
    s = sum(row)
    r = s - n / 2
    print(f"level={level:4d}, N={n:4d}, S={s:4d}, rho={s/n:.12f}, R={r}")
    print(" mask:", lucas_offsets(level)[:20], "..." if len(lucas_offsets(level)) > 20 else "")
    if n <= 32:
        print(" row:", row)
    print()

for level in [448, 512, 576, 1024, 2046, 2047]:
    summarize_level(level)

assert binary_trace["N"][448] == 1600
assert binary_trace["S"][448] == 800
assert binary_trace["R"][448] == 0.0

assert binary_trace["N"][512] == 1536
assert binary_trace["S"][512] == 764
assert binary_trace["R"][512] == -4.0

assert binary_trace["N"][576] == 1472
assert binary_trace["S"][576] == 749
assert binary_trace["R"][576] == 13.0

assert trajectory_parity[2046] == [1, 1]
assert trajectory_parity[2047] == [0]

print("Structural pins verified.")


## 7. Dyadic Terminal Tomography Theorem

For:

$$
N=2^m
$$

and:

$$
\ell_k=N-2^k,\qquad k=0,1,\ldots,m-1,
$$

the terminal row is:

$$
x_i^{(N-2^k)}
=
\bigoplus_{q=0}^{2^{m-k}-1}
x_{i+q2^k}^{(0)},
\qquad
0\le i<2^k.
$$

For $N=2048=2^{11}$, use $k=0,\ldots,10$.

Do **not** include $k=11$, because that is the identity row $\ell=0$.


In [ ]:
# Cell 8 — Verify dyadic terminal theorem

def dyadic_terminal_row_from_seed(seed_bits, k):
    modulus = 1 << k
    out = []
    for i in range(modulus):
        v = 0
        for q in range(i, len(seed_bits), modulus):
            v ^= seed_bits[q]
        out.append(v)
    return out

m = int(math.log2(N))
assert 2 ** m == N

dyadic_summary = []

for k in range(0, m):
    level = N - (1 << k)
    theorem_row = dyadic_terminal_row_from_seed(seed_parity, k)
    actual_row = trajectory_parity[level]
    ok = theorem_row == actual_row
    dyadic_summary.append((k, level, len(actual_row), ok))
    assert ok

print("Dyadic terminal levels:")
print(f"{'k':>3} {'level':>6} {'row_len':>8} {'verified':>10}")
print("-" * 34)
for k, level, row_len, ok in dyadic_summary:
    print(f"{k:3d} {level:6d} {row_len:8d} {str(ok):>10}")
    if k <= 4:
        print("    row:", trajectory_parity[level])

print("Dyadic terminal theorem verified.")


## 8. Correct GF(2) Rank Engine

The corrected dyadic equation count is:

$$
\sum_{k=0}^{10}2^k=2047.
$$

Correct rank:

$$
\operatorname{rank}(C_{\mathrm{dyadic}})=1024.
$$

Correct combined lock:

$$
\operatorname{rank}
\begin{bmatrix}
C_{\mathrm{dyadic}}\\
C_{448}
\end{bmatrix}
=1600.
$$


In [ ]:
# Cell 9 — Rank engine

def masks_for_level(N, level):
    offsets = lucas_offsets(level)
    masks = []
    for i in range(N - level):
        mask = 0
        for j in offsets:
            mask |= 1 << (i + j)
        masks.append(mask)
    return masks

def gf2_rank_int(rows):
    basis = {}
    for row in rows:
        x = int(row)
        while x:
            pivot = x.bit_length() - 1
            if pivot in basis:
                x ^= basis[pivot]
            else:
                basis[pivot] = x
                break
    return len(basis)

print("GF(2) integer-bitmask rank engine ready.")


In [ ]:
# Cell 10 — Compute correct ranks

if RUN_FULL_RANK:
    t0 = time.time()

    dyadic_masks = []
    for k in range(0, m):
        level = N - (1 << k)
        dyadic_masks.extend(masks_for_level(N, level))

    masks_448 = masks_for_level(N, 448)
    masks_512 = masks_for_level(N, 512)
    masks_576 = masks_for_level(N, 576)

    rank_dyadic = gf2_rank_int(dyadic_masks)
    rank_448 = gf2_rank_int(masks_448)
    rank_512 = gf2_rank_int(masks_512)
    rank_576 = gf2_rank_int(masks_576)
    rank_dyadic_448 = gf2_rank_int(dyadic_masks + masks_448)
    rank_dyadic_512 = gf2_rank_int(dyadic_masks + masks_512)
    rank_dyadic_576 = gf2_rank_int(dyadic_masks + masks_576)
    rank_dyadic_448_576 = gf2_rank_int(dyadic_masks + masks_448 + masks_576)

    elapsed = time.time() - t0

    print(f"Rank computation time: {elapsed:.3f}s")
    print("Dyadic equation count:", len(dyadic_masks))
    print("rank(dyadic):", rank_dyadic)
    print("rank(448):", rank_448)
    print("rank(512):", rank_512)
    print("rank(576):", rank_576)
    print("rank(dyadic + 448):", rank_dyadic_448)
    print("rank(dyadic + 512):", rank_dyadic_512)
    print("rank(dyadic + 576):", rank_dyadic_576)
    print("rank(dyadic + 448 + 576):", rank_dyadic_448_576)
    print("remaining degrees after dyadic+448:", N - rank_dyadic_448)

    assert len(dyadic_masks) == 2047
    assert rank_dyadic == 1024
    assert rank_448 == 1600
    assert rank_512 == 1536
    assert rank_576 == 1472
    assert rank_dyadic_448 == 1600
    assert rank_dyadic_512 == 1536
    assert rank_dyadic_576 == 1472
    assert N - rank_dyadic_448 == 448
else:
    print("RUN_FULL_RANK=False")


## 9. Residue Count Lock

Correct finite-cone full run:

$$
R=0:\ 44,
\qquad
R\neq0:\ 2004.
$$

Interior window:

$$
20\le\ell<1800
$$

gives:

$$
R=0:\ 30,
\qquad
R\neq0:\ 1750.
$$


In [ ]:
# Cell 11 — Residue counts

zero_R_levels = binary_trace["locks"]
nonzero_R_levels = [i for i, R in enumerate(binary_trace["R"]) if R != 0]

print("Full run:")
print("  total levels:", len(binary_trace["R"]))
print("  R=0:", len(zero_R_levels))
print("  R!=0:", len(nonzero_R_levels))
print("  check:", len(zero_R_levels) + len(nonzero_R_levels))
print("  first 15 locks:", zero_R_levels[:15])
print("  last 20 locks:", zero_R_levels[-20:])

interior_start, interior_end = 20, 1800
interior_R = binary_trace["R"][interior_start:interior_end]
interior_zero = [interior_start + i for i, R in enumerate(interior_R) if R == 0]
interior_nonzero = [interior_start + i for i, R in enumerate(interior_R) if R != 0]

print()
print(f"Interior window {interior_start} <= level < {interior_end}:")
print("  window length:", len(interior_R))
print("  R=0:", len(interior_zero))
print("  R!=0:", len(interior_nonzero))
print("  check:", len(interior_zero) + len(interior_nonzero))

assert len(zero_R_levels) == 44
assert len(nonzero_R_levels) == 2004
assert len(interior_zero) == 30
assert len(interior_nonzero) == 1750


## 10. Mark-9 / $H=\pi/9$ Loop Ladder Audit

Read:

$$
H=\frac{\pi}{9}
$$

as a phase unit:

$$
9H=\pi,
\qquad
18H=2\pi.
$$

The fold-lattice phase ladder uses:

$$
\ell_n=64n,
\qquad
n=1,\ldots,18.
$$

The Mark-9 pin is:

$$
\ell_9=576.
$$


In [ ]:
# Cell 12 — 64n loop ladder audit

print("64n Loop Ladder Audit")
print("=" * 80)
print(f"{'n':>3} {'level':>6} {'binary':>14} {'mask_size':>10} {'N':>6} {'S':>6} {'R':>8} {'rho':>10}")
print("-" * 80)

loop_ladder = []

for n in range(1, 19):
    level = 64 * n
    offsets = lucas_offsets(level)
    Nl = int(binary_trace["N"][level])
    Sl = int(binary_trace["S"][level])
    Rl = float(binary_trace["R"][level])
    rho = float(binary_trace["rho"][level])
    loop_ladder.append({
        "n": n,
        "level": level,
        "binary": bin(level),
        "mask_size": len(offsets),
        "offsets": offsets,
        "N": Nl,
        "S": Sl,
        "R": Rl,
        "rho": rho,
    })
    mark = "  <-- MARK-9" if n == 9 else ""
    print(f"{n:3d} {level:6d} {bin(level)[2:]:>14} {len(offsets):10d} {Nl:6d} {Sl:6d} {Rl:8.1f} {rho:10.6f}{mark}")

print()
print("Mark-9 offsets:", loop_ladder[8]["offsets"])

assert loop_ladder[8]["level"] == 576
assert loop_ladder[8]["offsets"] == [0, 64, 512, 576]
assert loop_ladder[8]["N"] == 1472
assert loop_ladder[8]["S"] == 749
assert loop_ladder[8]["R"] == 13.0


## 11. Shift-XOR Reverse Primitive

For one shift factor:

$$
y_i=x_i\oplus x_{i+s}.
$$

Given the first $s$ boundary bits:

$$
x_{i+s}=x_i\oplus y_i.
$$

For:

$$
448=256+128+64,
$$

boundary entropy is:

$$
256+128+64=448.
$$


In [ ]:
# Cell 13 — Shift-XOR reverse primitive

def reverse_one_shift_xor(y, shift, boundary):
    assert len(boundary) == shift
    x = list(boundary) + [0] * len(y)
    for i, yi in enumerate(y):
        x[i + shift] = x[i] ^ yi
    return x

x_true = [1,0,1,1,0,0,1,0,1,1]
shift = 3
y = [x_true[i] ^ x_true[i+shift] for i in range(len(x_true)-shift)]
x_rec = reverse_one_shift_xor(y, shift, x_true[:shift])

print("x_true:  ", x_true)
print("y:       ", y)
print("boundary:", x_true[:shift])
print("x_rec:   ", x_rec)

assert x_rec == x_true


## 12. Small-$N$ Trace-Sufficient Decoder Demo

A full $2048$ decode requires pseudo-Boolean / SAT / MILP filtering over the remaining $448$ variables.

This small demo uses:

$$
N_{\mathrm{demo}}=16
$$

and shows:

$$
2^{16}
\rightarrow
\text{linear candidates}
\rightarrow
\text{weight-trace collapse}.
$$


In [ ]:
# Cell 14 — Small-N decoder demo

def int_to_bits(n, width):
    return [(n >> i) & 1 for i in range(width)]

def bits_to_int(bits):
    out = 0
    for i, bit in enumerate(bits):
        if bit:
            out |= 1 << i
    return out

def dyadic_levels_for_N(n):
    mm = int(math.log2(n))
    assert 2 ** mm == n
    return [n - (1 << k) for k in range(mm)]

def linear_signature(bits, levels):
    return tuple(tuple(row_from_lucas(bits, level)) for level in levels)

def weight_signature(bits):
    return tuple(sum(r) for r in build_xor_trajectory(bits))

N_demo = 16
seed_demo = seed_parity[:N_demo]
seed_demo_int = bits_to_int(seed_demo)

pin_demo = 7
linear_levels_demo = dyadic_levels_for_N(N_demo) + [pin_demo]

target_linear = linear_signature(seed_demo, linear_levels_demo)
target_weight = weight_signature(seed_demo)

linear_candidates = []
weight_candidates = []

for n in range(1 << N_demo):
    bits = int_to_bits(n, N_demo)
    if linear_signature(bits, linear_levels_demo) == target_linear:
        linear_candidates.append(n)
        if weight_signature(bits) == target_weight:
            weight_candidates.append(n)

print("N_demo:", N_demo)
print("Seed demo bits:", seed_demo)
print("Seed demo int:", seed_demo_int)
print("Linear levels:", linear_levels_demo)
print("Linear candidate count:", len(linear_candidates))
print("Weight-trace candidate count:", len(weight_candidates))
print("Weight candidates:", weight_candidates)

assert seed_demo_int in linear_candidates
assert seed_demo_int in weight_candidates
assert len(weight_candidates) == 1
assert weight_candidates[0] == seed_demo_int


## 13. Visual Diagnostics


In [ ]:
# Cell 15 — Visual diagnostics

if RUN_VISUALS:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    axes[0,0].plot(decimal_S, linewidth=1)
    for level in [13, 448, 512, 576]:
        axes[0,0].axvline(level, linestyle="--", linewidth=1)
    axes[0,0].set_title("Decimal Value-Channel Sum")
    axes[0,0].set_xlabel("Level ℓ")
    axes[0,0].set_ylabel("S_ℓ^(10)")

    R = binary_trace["R"]
    axes[0,1].plot(R, linewidth=0.7)
    axes[0,1].scatter(binary_trace["locks"], [0]*len(binary_trace["locks"]), s=12)
    for level in [448, 512, 576]:
        axes[0,1].axvline(level, linestyle="--", linewidth=1)
    axes[0,1].axhline(0, linewidth=1)
    axes[0,1].set_title("Binary Residue Wave R_ℓ")
    axes[0,1].set_xlabel("Level ℓ")
    axes[0,1].set_ylabel("R_ℓ")

    segment = R[20:1800]
    axes[1,0].hist(segment, bins=50, edgecolor="black")
    axes[1,0].axvline(0, linestyle="--", linewidth=1)
    axes[1,0].set_title("Residue Distribution (20 ≤ ℓ < 1800)")
    axes[1,0].set_xlabel("R_ℓ")
    axes[1,0].set_ylabel("Frequency")

    ns = [d["n"] for d in loop_ladder]
    rs = [d["R"] for d in loop_ladder]
    axes[1,1].bar(ns, rs)
    axes[1,1].axhline(0, linewidth=1)
    axes[1,1].axvline(9, linestyle="--", linewidth=1)
    axes[1,1].set_title("64n Loop Ladder Residues")
    axes[1,1].set_xlabel("n in ℓ=64n")
    axes[1,1].set_ylabel("R_ℓ")

    plt.tight_layout()
    plt.show()
else:
    print("RUN_VISUALS=False")


In [ ]:
# Cell 16 — Parity shadow crop

if RUN_VISUALS:
    max_levels = 700
    max_cols = 700

    img = np.ones((max_levels, max_cols), dtype=float)

    for level in range(max_levels):
        row = trajectory_parity[level][:max_cols]
        for i, bit in enumerate(row):
            img[level, i] = 0.0 if bit else 1.0

    plt.figure(figsize=(12, 10))
    plt.imshow(img, cmap="gray", aspect="auto", interpolation="nearest")
    for level in [448, 512, 576, 640]:
        plt.axhline(level, linestyle="--", linewidth=0.8)
        plt.text(max_cols + 5, level, f"ℓ={level}", va="center")
    plt.title("Finite-Cone Parity Shadow: Rule 90 / Lucas Masks")
    plt.xlabel("Position")
    plt.ylabel("Level ℓ")
    plt.show()
else:
    print("RUN_VISUALS=False")


## 14. Final Validation


In [ ]:
# Cell 17 — Final validation summary

checks = {
    "skip_leading_3": pi_fractional[:20] == "14159265358979323846",
    "decimal_S0": int(decimal_S[0]) == 9338,
    "decimal_S13": int(decimal_S[13]) == 1092,
    "parity_S0": int(binary_trace["S"][0]) == 1034,
    "parity_shadow_exact": violations == 0,
    "finite_cone_levels": len(trajectory_parity) == 2048 and len(trajectory_parity[-1]) == 1,
    "level_448_lock": int(binary_trace["N"][448]) == 1600 and int(binary_trace["S"][448]) == 800 and float(binary_trace["R"][448]) == 0.0,
    "level_512_lock": int(binary_trace["N"][512]) == 1536 and int(binary_trace["S"][512]) == 764 and float(binary_trace["R"][512]) == -4.0,
    "mark9_576_lock": int(binary_trace["N"][576]) == 1472 and int(binary_trace["S"][576]) == 749 and float(binary_trace["R"][576]) == 13.0,
    "dyadic_theorem": all(ok for _, _, _, ok in dyadic_summary),
    "full_R_counts": len(zero_R_levels) == 44 and len(nonzero_R_levels) == 2004,
    "interior_R_counts": len(interior_zero) == 30 and len(interior_nonzero) == 1750,
    "smallN_decoder": len(weight_candidates) == 1 and weight_candidates[0] == seed_demo_int,
}

if RUN_FULL_RANK:
    checks.update({
        "rank_dyadic_1024": rank_dyadic == 1024,
        "rank_448_1600": rank_448 == 1600,
        "rank_dyadic_448_1600": rank_dyadic_448 == 1600,
        "nullity_448": (N - rank_dyadic_448) == 448,
    })

print("FINAL VALIDATION")
print("=" * 80)
for name, ok in checks.items():
    print(f"{name:30s}: {ok}")

assert all(checks.values())

print()
print("Dyadic Tomography Reverse Engine v2 FIXED validation complete.")
print("Core executable collapse: 2048 -> 1600 -> 448")
print("Mark-9 phase checkpoint: level 576, mask [0,64,512,576], R=13")


## Final Formula Stack

$$
\Delta:
\quad
d_i^{(\ell+1)}
=
|d_{i+1}^{(\ell)}-d_i^{(\ell)}|
$$

$$
\oplus:
\quad
x_i^{(\ell+1)}
=
x_i^{(\ell)}\oplus x_{i+1}^{(\ell)}
$$

$$
↻:
\quad
x^{(\ell)}
=
(I+E)^\ell x^{(0)}
$$

$$
\bot:
\quad
x_i^{(\ell)}
=
\bigoplus_{j\subseteq\ell}x_{i+j}^{(0)}
$$

$$
\Psi:
\quad
C_{\mathrm{dyadic}}x=y,
\qquad
\operatorname{rank}
\begin{bmatrix}
C_{\mathrm{dyadic}}\\
C_{448}
\end{bmatrix}
=1600,
\qquad
2048-1600=448
$$

$$
H=\frac{\pi}{9},
\qquad
9H=\pi,
\qquad
\ell_9=9\cdot64=576.
$$

Final lock:

$$
\boxed{
\text{The old cyclic prototype is fixed into the finite-cone fractional-}\pi\text{ FOLD-TOMO engine.}
}
$$
